# Step one: Local

We'll use a local [Ollama](https://ollama.com) instance running [phi4-mini](https://ollama.com/library/phi4-mini) since it's fast, small and has built-in tool calling capabilities.

This will allow us to create our tools and iterate **fast** on our prototype before switching to OpenAI.

In [2]:
!uv pip install langchain langchain-ollama 

Audited 2 packages in 19ms


In [3]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="phi4-mini")
prompt = "Hi!"
response = model.invoke(prompt)
print(response)

content="Hello there! How can I assist you today? Whether you're looking for information, help with a specific task or just chatting about something interesting—I’m here to listen and aid in any way that I'm able to. What’s on your mind lately?" additional_kwargs={} response_metadata={'model': 'phi4-mini', 'created_at': '2025-06-21T20:34:51.625547Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15322280541, 'load_duration': 3749992750, 'prompt_eval_count': 5, 'prompt_eval_duration': 2808409708, 'eval_count': 50, 'eval_duration': 8760536458, 'model_name': 'phi4-mini'} id='run--d273243e-0102-4a80-85ef-4f28ae7fd7ba-0' usage_metadata={'input_tokens': 5, 'output_tokens': 50, 'total_tokens': 55}


## Sample tool call

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field

class AdditionInput(BaseModel):
    x: int = Field(..., description="The first number to add.")
    y: int = Field(..., description="The second number to add.")

@tool("add-tool", args_schema=AdditionInput, return_direct=True)
def add(x: int, y: int) -> int:
    """Adds two numbers together."""
    return x + y

In [12]:
from langchain_core.messages import HumanMessage, BaseMessage
from typing import List

model_with_tool = model.bind_tools([add])
messages: List[BaseMessage] = [HumanMessage(content="What is 2 + 3?")]
response = model_with_tool.invoke(messages)
messages.append(response)
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'phi4-mini', 'created_at': '2025-06-21T20:53:00.407424Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8013907084, 'load_duration': 73218834, 'prompt_eval_count': 80, 'prompt_eval_duration': 1022548708, 'eval_count': 35, 'eval_duration': 6913757833, 'model_name': 'phi4-mini'}, id='run--36b52b85-0b7e-4630-a15f-fad10c4bb4a5-0', tool_calls=[{'name': 'add', 'args': {'x': 2, 'y': 3}, 'id': '55ce5274-fb14-408c-bb71-320866d2df1d', 'type': 'tool_call'}], usage_metadata={'input_tokens': 80, 'output_tokens': 35, 'total_tokens': 115})

In [ ]:
for tool_call in response.tool_calls:
    tool = {"add": add}[tool_call["name"]]
    tool_response = tool.invoke(tool_call)
    messages.append(tool_response)
    print(tool_response)

content='5' name='add' tool_call_id='55ce5274-fb14-408c-bb71-320866d2df1d'


In [15]:
response = model_with_tool.invoke(messages)
response

AIMessage(content="That's correct! The sum of 2 and 3 equals 5. If you have any more questions or need further assistance, feel free to ask!", additional_kwargs={}, response_metadata={'model': 'phi4-mini', 'created_at': '2025-06-21T20:55:47.178637Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5207067792, 'load_duration': 46315709, 'prompt_eval_count': 103, 'prompt_eval_duration': 1652922083, 'eval_count': 32, 'eval_duration': 3496643708, 'model_name': 'phi4-mini'}, id='run--d3a22033-43dd-4ce5-b853-ab34d50645cd-0', usage_metadata={'input_tokens': 103, 'output_tokens': 32, 'total_tokens': 135})